## Dự án lớn bắt đầu!!

# Dự án Capstone "THE PRICE IS RIGHT"

Tuần này - xây dựng một mô hình dự đoán giá của một sản phẩm từ mô tả, dựa trên dữ liệu được scrape từ Amazon

# Trình tự

NGÀY 1: Thu thập và tinh chỉnh dữ liệu  
NGÀY 2: Tiền xử lý dữ liệu  
NGÀY 3: Đánh giá, baseline, Machine Learning truyền thống  
NGÀY 4: Deep Learning và LLM  
NGÀY 5: Fine-tuning một mô hình frontier  

## NGÀY 1: Thu thập và tinh chỉnh dữ liệu

Hôm nay chúng ta sẽ làm sạch và chọn lọc dữ liệu

Tập dữ liệu ở đây:  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

Và thư mục chứa tất cả các tập dữ liệu sản phẩm ở đây:  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Giá trị kinh doanh của việc tinh chỉnh dữ liệu</h2>
            <span style="color:#181;">Việc tinh chỉnh dữ liệu có thể được xem như phần công việc ít lộ liễu hơn của một nhà khoa học dữ liệu. Tôi cho rằng đó là vô nghĩa!
            Đây chính là nơi khoa học diễn ra - điều gì có thể lôi cuốn hơn thế? Nghiên cứu và phát triển với tập dữ liệu của bạn
            thường có tác động lớn hơn nhiều so với việc 'tối ưu siêu tham số' thời thượng mà ta làm sau này.
            Vì vậy: hãy sẵn sàng dành thời gian cho Chất lượng Dữ liệu.</span>
        </td>
    </tr>
</table>

In [ ]:
# Import các thư viện cần thiết cho việc tải dữ liệu và xử lý đồ thị
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
from pricer.items import Item
from pricer.parser import parse

# Tải biến môi trường từ file .env để có HF_TOKEN
load_dotenv(override=True)


In [ ]:
# Đăng nhập Hugging Face bằng token đã lưu trong biến môi trường
# Nếu bạn thấy thông báo 'Note about Environment variable being set', hãy bỏ qua
# vì đó không phải lỗi nghiêm trọng.
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)


## Tải tập dữ liệu của chúng ta

Trong ô kế tiếp, chúng ta sẽ tải dữ liệu từ Hugging Face.

Nếu gặp lỗi như "trust_remote_code is no longer supported" (không còn hỗ trợ remote code), thì hãy chạy lệnh sau trong một ô mới: `!uv add --upgrade datasets==3.6.0` và sau đó khởi động lại Kernel, rồi thử lại.


In [ ]:
# Tải tập dữ liệu sản phẩm category 'Appliances' từ Hugging Face
# split='full' nghĩa là lấy toàn bộ dữ liệu, không chỉ một phần nhỏ
# trust_remote_code=True để cho phép code chạy từ remote nếu cần thiết
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)


In [ ]:
# In số lượng phần tử trong dataset Appliances
print(f"Số lượng Appliances: {len(dataset):,}")


In [ ]:
# Investigate a particular datapoint

dataset[6]


In [ ]:
# Tìm sản phẩm có giá cao nhất trong dataset và in ra tên + giá
# tqdm giúp hiển thị tiến độ khi duyệt qua từng datapoint
max_price = 0
max_item = None

for datapoint in tqdm(dataset):
    try:
        # Chuyển giá từ text sang float để so sánh
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except ValueError:
        # Bỏ qua các hàng có giá không hợp lệ hoặc thiếu thông tin
        pass

print(f"Sản phẩm đắt nhất là {max_item['title']} với giá {max_price:,.2f}")


Đây là cái gần nhất mà tôi tìm được - trông giống như đang được bán với giá hời!!

https://www.amazon.com/TurboChef-Electric-Countertop-Microwave-Convection/dp/B01D05U9NO/


In [ ]:
# Chuyển dữ liệu thô thành các đối tượng Item nếu giá nằm trong khoảng $1-$1000
# và có đủ thông tin chi tiết để sử dụng cho dự án
items = [parse(datapoint, "Appliances") for datapoint in tqdm(dataset)]
items = [item for item in items if item is not None]
print(f"Có {len(items):,} sản phẩm hợp lệ từ tổng {len(dataset):,} datapoints")


In [ ]:
# Xem đối tượng Item đầu tiên để kiểm tra cấu trúc dữ liệu đã được chuẩn hóa
items[0]


In [ ]:
# In toàn bộ nội dung đã được làm sạch của Item đầu tiên
print(items[0].full)


In [ ]:
# Tạo hai danh sách:
# - prices: giá của từng sản phẩm
# - lengths: độ dài văn bản mô tả của từng sản phẩm
prices = [item.price for item in items]
lengths = [len(item.full) for item in items]


In [ ]:
# Vẽ biểu đồ phân bố độ dài text của các sản phẩm
plt.figure(figsize=(15, 6))
plt.title(f"Độ dài văn bản: Trung bình {sum(lengths)/len(lengths):,.0f} và cao nhất {max(lengths):,}\n")
plt.xlabel('Độ dài (ký tự)')
plt.ylabel('Số lượng')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 6000, 100))
plt.show()


In [ ]:
# Tìm item có văn bản dài nhất để xem dữ liệu cực đoan
max_length = max(lengths)
max_length_item = items[lengths.index(max_length)]
print(max_length_item.full)


In [ ]:
# Vẽ biểu đồ phân bố giá sản phẩm
plt.figure(figsize=(15, 6))
plt.title(f"Giá sản phẩm: Trung bình {sum(prices)/len(prices):,.2f} và cao nhất {max(prices):,}\n")
plt.xlabel('Giá ($)')
plt.ylabel('Số lượng')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()


In [ ]:
# In nội dung hoàn chỉnh của item thứ 3 để xem chi tiết sản phẩm
print(items[3].full)


In [ ]:
# Dùng ItemLoader để tải nhiều category khác nhau một cách tự động
from pricer.loaders import ItemLoader
loader = ItemLoader("Appliances")
items = loader.load()


In [ ]:
# Danh sách các category dữ liệu mà ta sẽ làm sạch và kết hợp lại
# Mỗi category đại diện cho một nhóm sản phẩm trên Amazon
# Ví dụ: Automotive, Electronics, Office_Products, ...
dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]


In [ ]:
# Tải và gộp tất cả item từ nhiều category khác nhau
items = []
for dataset_name in dataset_names:
    loader = ItemLoader(dataset_name)
    items.extend(loader.load())


In [ ]:
# In tổng số item sau khi ghép dữ liệu từ nhiều category
print(f"Tổng cộng có {len(items):,} item")


In [ ]:
# Xem một item ở vị trí 1000 để kiểm tra chất lượng dữ liệu đã tổng hợp
items[1000]


In [ ]:
# Trộn ngẫu nhiên dữ liệu để khởi động việc loại trùng lặp
# Sau đó loại bỏ các item có tiêu đề hoặc full text trùng lặp
random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"Sau khi loại bỏ trùng lặp, còn {len(items):,} item")


In [ ]:
# Vẽ lại phân bố chiều dài văn bản sau khi đã clean và dedupe
lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Độ dài văn bản: Trung bình {sum(lengths)/len(lengths):,.1f} và tối đa {max(lengths):,}\n")
plt.xlabel('Độ dài (ký tự)')
plt.ylabel('Số lượng')
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()


In [ ]:
# Vẽ biểu đồ phân bố giá sau khi xử lý dữ liệu cuối cùng
prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Giá: Trung bình {sum(prices)/len(prices):,.1f} và cao nhất {max(prices):,}\n")
plt.xlabel('Giá ($)')
plt.ylabel('Số lượng')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()


In [ ]:
# Đếm số lượng item theo từng category để biết dữ liệu có bị lệch không
from collections import Counter
category_counts = Counter([item.category for item in items])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('Số lượng trong từng category')
plt.xlabel('Category')
plt.ylabel('Số lượng')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()


In [ ]:
# Chọn một sample 820k item từ dữ liệu tổng thể
# Mục đích: tạo tập dữ liệu cân bằng hơn theo phân bố giá và category
np.random.seed(42)

SIZE = 820_000

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

# Trọng số theo giá, với khối lượng lớn hơn ở các category giá rẻ hơn
w = p**2
w[categories == "Tools_and_Home_Improvement"] *= 0.5
w[categories == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]


In [ ]:
# Vẽ phân bố giá của sample vừa chọn
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Giá trong sample: Trung bình {sum(prices)/len(prices):,.1f} thấp nhất {min(prices):,} và cao nhất {max(prices):,}\n")
plt.xlabel('Giá ($)')
plt.ylabel('Số lượng')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()


In [ ]:
# Trộn lại sample lần cuối trước khi dùng làm dataset cuối cùng
random.seed(42)
random.shuffle(sample)


In [ ]:
# Vẽ lại phân bố giá sau khi trộn dataset
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Giá sau khi trộn: Trung bình {sum(prices)/len(prices):,.1f} thấp nhất {min(prices):,} và cao nhất {max(prices):,}\n")
plt.xlabel('Giá ($)')
plt.ylabel('Số lượng')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()


In [ ]:
# Đếm số lượng item theo category trong sample sau khi chọn mẫu
from collections import Counter
category_counts = Counter([item.category for item in sample])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

# Biểu đồ cột theo category
plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('Số lượng theo từng category')
plt.xlabel('Category')
plt.ylabel('Số lượng')

plt.xticks(rotation=30, ha='right')

# Thêm nhãn số lượng trên đầu mỗi cột
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

# Hiển thị biểu đồ
plt.show()


In [ ]:
# Automotive vẫn chiếm ưu thế, nhưng đã cải thiện một chút
# Để xem góc nhìn khác, ta vẽ biểu đồ hình tròn

plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct='%1.0f%%', startangle=90)

# Thêm vòng tròn ở giữa để tạo donut chart (tùy chọn)
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Category')

# Tỉ lệ bằng nhau để đảm bảo pie vẽ thành hình tròn
plt.axis('equal')  

plt.show()


In [ ]:
# Giá có thay đổi theo độ dài văn bản hay không?

sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

# Tạo scatter plot
plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")

# Thêm nhãn và tiêu đề
plt.xlabel('Kích thước văn bản')
plt.ylabel('Giá')
plt.title('Có phải giá có tương quan đơn giản với độ dài text?')

# Hiển thị biểu đồ
plt.show()


In [ ]:
# Giá có thay đổi theo trọng lượng hay không?

oz = [item.weight for item in sample]
prices = [item.price for item in sample]

# Tạo scatter plot
plt.figure(figsize=(15, 8))
plt.scatter(oz, prices, s=0.2, color="darkorange")

# Thêm nhãn và tiêu đề
plt.xlabel('Trọng lượng (ounce)')
plt.ylabel('Giá')
plt.xlim(0, 400)
plt.title('Có phải giá có tương quan đơn giản với trọng lượng?')

# Hiển thị biểu đồ
plt.show()


## Bây giờ đẩy tập dữ liệu này lên Hugging Face Hub

Thay username bằng tên người dùng HF của bạn nếu bạn đã tạo tập dữ liệu riêng

Hoặc, bỏ qua ô này và bạn có thể tải tập dữ liệu của tôi vào ngày mai!


In [ ]:
# Thiết lập username Hugging Face của bạn
# Nếu bạn đã tạo dataset riêng, hãy thay username bằng tên của bạn
username = "ed-donner"
full = f"{username}/items_raw_full"
lite = f"{username}/items_raw_lite"

# Tạo tập train/val/test chính
train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

# Đẩy dataset đầy đủ lên Hugging Face Hub
Item.push_to_hub(full, train, val, test)

# Tạo các phiên bản nhỏ hơn để dễ test nhanh hơn
train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

# Đẩy dataset nhỏ lên Hub
Item.push_to_hub(lite, train_lite, val_lite, test_lite)


## Lưu ý phụ

Nếu bạn thích sự đa dạng của các màu mà matplotlib có thể sử dụng trong biểu đồ, bạn nên đánh dấu trang này:

https://matplotlib.org/stable/gallery/color/named_colors.html
